# NCKH-UAV: Adversarial Camouflage Attack Pipeline — v5**Changes from v4:**- **Fixed DIOR processing**: VOC XML → YOLO format conversion (was 0 images, now ~5000+)- Dataset now includes both DOTA slices + DIOR images → larger, more diverse training set**Pipeline:** Data prep → Train surrogate (YOLOv8n) → DE attack → Evaluate → (Phase 6: transfer to RT-DETR)

In [ ]:
# ── Cell 1: Environment & GPU ─────────────────────────────
import os, sys, shutil, subprocess, time
from pathlib import Path

# Kaggle dataset mount points
DOTA_INPUT  = '/kaggle/input/datasets/thuyen001/dota-raw'
DIOR_INPUT  = '/kaggle/input/datasets/thuyen001/dior-raw/dior_raw'
CODE_SRC    = '/kaggle/input/datasets/thuyen001/nckh-uav-code/nckh-uav-code'

WORK = Path('/kaggle/working')
DATA = WORK / 'data'

# Copy source tree
dst_src = WORK / 'src'
if dst_src.exists():
    shutil.rmtree(dst_src)
shutil.copytree(os.path.join(CODE_SRC, 'src'), str(dst_src))
shutil.copy2(os.path.join(CODE_SRC, 'configs/base_config.yaml'),
             str(WORK / 'base_config.yaml'))
sys.path.insert(0, str(WORK))

# GPU info
import torch
print(f'PyTorch {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
for i in range(torch.cuda.device_count()):
    print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')
print()


In [ ]:
# ── Cell 2: Install packages ──────────────────────────────
!pip install -q ultralytics opencv-python-headless pyyaml tqdm
import ultralytics
print(f'Ultralytics {ultralytics.__version__}')


In [ ]:
# ── Cell 3: Create symlinks & run diagnostics ─────────────
from pathlib import Path
import os

# ── DOTA raw ──
dota_img_src = os.path.join(DOTA_INPUT, 'images')
dota_lbl_src = os.path.join(DOTA_INPUT, 'labelTxt')
dota_dst = DATA / 'DOTA' / 'raw'
dota_dst.mkdir(parents=True, exist_ok=True)

for name, src in [('images', dota_img_src), ('labelTxt', dota_lbl_src)]:
    link = dota_dst / name
    if not link.exists():
        os.symlink(src, str(link))

# ── DIOR raw (FIXED: actual folder names are JPEGImages & Annotations) ──
dior_dst = DATA / 'DIOR' / 'raw'
dior_dst.mkdir(parents=True, exist_ok=True)

for folder_name in ['JPEGImages', 'Annotations']:
    link = dior_dst / folder_name
    src  = os.path.join(DIOR_INPUT, folder_name)
    if os.path.isdir(src) and not link.exists():
        os.symlink(src, str(link))
        print(f'  DIOR symlink: {folder_name} → OK')
    elif not os.path.isdir(src):
        print(f'  DIOR WARNING: {src} not found!')

# Quick check
n_dota_img = len(list(Path(dota_img_src).glob('*.png')))
n_dota_lbl = len(list(Path(dota_lbl_src).glob('*.txt')))
print(f'\nDOTA images: {n_dota_img}  labels: {n_dota_lbl}  matched: {min(n_dota_img,n_dota_lbl)} ✅')

# Check DIOR
dior_jpeg_dir = dior_dst / 'JPEGImages'
dior_ann_dir  = dior_dst / 'Annotations'
if dior_jpeg_dir.exists():
    import glob
    n_dior_img = len(glob.glob(str(dior_jpeg_dir / '*.jpg')))
    n_dior_xml = len(glob.glob(str(dior_ann_dir / '*.xml')))
    print(f'DIOR images: {n_dior_img}  XML labels: {n_dior_xml} ✅')
else:
    print('DIOR: JPEGImages not found — check DIOR_INPUT path')


In [ ]:
# ── Cell 4: Quick GPU test ────────────────────────────────
from ultralytics import YOLO
import numpy as np, cv2

dummy = np.random.randint(0, 255, (640, 640, 3), dtype=np.uint8)
tmp_path = '/tmp/gpu_test.jpg'
cv2.imwrite(tmp_path, dummy)

m = YOLO('yolov8n.pt')
r = m.predict(source=tmp_path, device='0', verbose=False)
print(f'Dummy GPU test passed — {len(r[0].boxes)} detections on noise image')


In [ ]:
# ── Cell 5: Process DOTA (PIL slicing) + DIOR (XML→YOLO) ─
import glob, cv2, numpy as np
from PIL import Image
from pathlib import Path
from tqdm import tqdm

Image.MAX_IMAGE_PIXELS = None  # large DOTA images

# ═══════════════════════════════════════════════════════════
# Part A: DOTA slicing (unchanged from v4)
# ═══════════════════════════════════════════════════════════
DOTA_CLASSES = {'plane': 0, 'small-vehicle': 1, 'large-vehicle': 1}
SLICE_SIZE = 1024
OVERLAP = 0.2

dota_out_img = DATA / 'DOTA' / 'processed' / 'images'
dota_out_lbl = DATA / 'DOTA' / 'processed' / 'labels'
dota_out_img.mkdir(parents=True, exist_ok=True)
dota_out_lbl.mkdir(parents=True, exist_ok=True)

dota_raw_imgs = sorted(glob.glob(str(DATA / 'DOTA' / 'raw' / 'images' / '*.png')))
dota_raw_lbls = str(DATA / 'DOTA' / 'raw' / 'labelTxt')

def parse_dota_label(label_path):
    anns = []
    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 9: continue
            cls_name = parts[8]
            if cls_name not in DOTA_CLASSES: continue
            coords = list(map(float, parts[:8]))
            xs = coords[0::2]
            ys = coords[1::2]
            anns.append((min(xs), min(ys), max(xs), max(ys), DOTA_CLASSES[cls_name]))
    return anns

total_slices = 0
read_ok = 0
read_fail = 0

for img_path in tqdm(dota_raw_imgs, desc='DOTA slicing'):
    stem = Path(img_path).stem
    lbl_path = os.path.join(dota_raw_lbls, stem + '.txt')
    if not os.path.isfile(lbl_path): continue
    anns = parse_dota_label(lbl_path)
    if len(anns) == 0: continue

    try:
        pil_img = Image.open(img_path)
        img_np = np.array(pil_img)
        if img_np.ndim == 2:
            img_np = cv2.cvtColor(img_np, cv2.COLOR_GRAY2BGR)
        elif img_np.shape[2] == 4:
            img_np = cv2.cvtColor(img_np, cv2.COLOR_RGBA2BGR)
        else:
            img_np = cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR)
        read_ok += 1
    except Exception as e:
        read_fail += 1
        continue

    h, w = img_np.shape[:2]
    stride = int(SLICE_SIZE * (1 - OVERLAP))

    for y0 in range(0, h, stride):
        for x0 in range(0, w, stride):
            x1 = min(x0 + SLICE_SIZE, w)
            y1 = min(y0 + SLICE_SIZE, h)
            x0a = x1 - SLICE_SIZE
            y0a = y1 - SLICE_SIZE
            if x0a < 0: x0a = 0
            if y0a < 0: y0a = 0

            slice_anns = []
            for (ax1, ay1, ax2, ay2, cls_id) in anns:
                ix1 = max(ax1, x0a)
                iy1 = max(ay1, y0a)
                ix2 = min(ax2, x1)
                iy2 = min(ay2, y1)
                iw = ix2 - ix1
                ih = iy2 - iy1
                if iw > 10 and ih > 10:
                    ow = ax2 - ax1
                    oh = ay2 - ay1
                    if (iw * ih) / max(ow * oh, 1) > 0.3:
                        sw = x1 - x0a
                        sh = y1 - y0a
                        cx = ((ix1 - x0a) + (ix2 - x0a)) / 2 / sw
                        cy = ((iy1 - y0a) + (iy2 - y0a)) / 2 / sh
                        bw = iw / sw
                        bh = ih / sh
                        slice_anns.append(f'{cls_id} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}')

            if len(slice_anns) == 0: continue

            slice_img = img_np[y0a:y1, x0a:x1]
            slice_name = f'{stem}_{x0a}_{y0a}'
            cv2.imwrite(str(dota_out_img / f'{slice_name}.jpg'), slice_img)
            with open(str(dota_out_lbl / f'{slice_name}.txt'), 'w') as f:
                f.write('\n'.join(slice_anns) + '\n')
            total_slices += 1

print(f'\nDOTA: {read_ok} read OK, {read_fail} failed, {total_slices} slices')

# ═══════════════════════════════════════════════════════════
# Part B: DIOR — Convert VOC XML → YOLO format (FIXED in v5)
# ═══════════════════════════════════════════════════════════
from src.data_prep.process_dior import main as process_dior_main

dior_proc_img = DATA / 'DIOR' / 'processed' / 'images'
dior_proc_lbl = DATA / 'DIOR' / 'processed' / 'labels'
dior_proc_img.mkdir(parents=True, exist_ok=True)
dior_proc_lbl.mkdir(parents=True, exist_ok=True)

dior_jpeg_dir = DATA / 'DIOR' / 'raw' / 'JPEGImages'
dior_ann_dir  = DATA / 'DIOR' / 'raw' / 'Annotations'

if dior_ann_dir.exists():
    n_dior = process_dior_main(
        input_img_dir=str(dior_jpeg_dir),
        input_xml_dir=str(dior_ann_dir),
        out_img_dir=str(dior_proc_img),
        out_lbl_dir=str(dior_proc_lbl),
        use_symlink=True,  # save disk on Kaggle
    )
    print(f'\nDIOR: {n_dior} images converted (XML → YOLO, symlinked)')
else:
    n_dior = 0
    print(f'DIOR: Annotations dir not found at {dior_ann_dir} — skipping')

print(f'\n══ Summary: DOTA {total_slices} slices + DIOR {n_dior} images ══')


In [ ]:
# ── Cell 6: Train/Val split + data.yaml ───────────────────
import random, shutil

SPLIT_RATIO = 0.8
SEED = 42

yolo_root = DATA / 'yolo_dataset'
for sub in ['train/images','train/labels','val/images','val/labels']:
    (yolo_root / sub).mkdir(parents=True, exist_ok=True)

def collect_pairs(img_dir, lbl_dir):
    pairs = []
    for ip in Path(img_dir).iterdir():
        if ip.suffix.lower() not in ('.jpg','.jpeg','.png','.bmp'): continue
        lp = Path(lbl_dir) / (ip.stem + '.txt')
        if lp.exists() and lp.stat().st_size > 5:
            pairs.append((str(ip), str(lp)))
    return pairs

dota_pairs = collect_pairs(dota_out_img, dota_out_lbl)
dior_pairs = collect_pairs(dior_proc_img, dior_proc_lbl)
all_pairs = dota_pairs + dior_pairs
print(f'DOTA {len(dota_pairs)} + DIOR {len(dior_pairs)} = {len(all_pairs)} pairs')

random.seed(SEED)
random.shuffle(all_pairs)
split = int(len(all_pairs) * SPLIT_RATIO)
train_pairs = all_pairs[:split]
val_pairs   = all_pairs[split:]
print(f'Train {len(train_pairs)} / Val {len(val_pairs)}')

def symlink_pairs(pairs, img_dst, lbl_dst):
    for ip, lp in pairs:
        di = Path(img_dst) / Path(ip).name
        dl = Path(lbl_dst) / Path(lp).name
        if not di.exists(): os.symlink(ip, str(di))
        if not dl.exists(): os.symlink(lp, str(dl))

symlink_pairs(train_pairs, yolo_root/'train/images', yolo_root/'train/labels')
symlink_pairs(val_pairs,   yolo_root/'val/images',   yolo_root/'val/labels')

# data.yaml
data_yaml = yolo_root / 'data.yaml'
data_yaml.write_text(
    f'path: {yolo_root}\n'
    f'train: train/images\n'
    f'val: val/images\n'
    f'nc: 2\n'
    f'names: ["aircraft", "vehicle"]\n'
)
print(f'data.yaml → {data_yaml}')


In [ ]:
# ── Cell 7: Train YOLOv8n surrogate ───────────────────────
from ultralytics import YOLO

EPOCHS = 50
IMGSZ  = 1024
BATCH  = 4

model = YOLO('yolov8n.pt')
results = model.train(
    data=str(data_yaml),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device='0,1',
    project=str(WORK / 'runs/detect'),
    name='surrogate_train',
    exist_ok=True,
    patience=10,
    save=True,
    save_period=10,
    plots=True,
    verbose=True,
)

# Locate best weights
TRAIN_DIR = WORK / 'runs/detect/surrogate_train'
BEST_PT   = TRAIN_DIR / 'weights' / 'best.pt'
print(f'\n✅ Training complete.  Best weights: {BEST_PT}')
print(f'   exists = {BEST_PT.exists()}')


In [ ]:
# ── Cell 8: ★ Real DE Attack with Trained Surrogate ★ ─────
import sys, os
sys.path.insert(0, str(WORK))

from src.attack.de_attack_real import DEAttackReal, plot_convergence, plot_patch, save_attack_examples

# ── Config ─────────────────────────────────────────────────
BEST_PT    = WORK / 'runs/detect/surrogate_train/weights/best.pt'
VAL_IMAGES = DATA / 'yolo_dataset' / 'val' / 'images'
VAL_LABELS = DATA / 'yolo_dataset' / 'val' / 'labels'

PATCH_SIZE   = 32        # 32×32 RGB
NUM_PATCHES  = 1         # 1 = single centre patch; 5 = multi-mini-patch (DeMPAA)
POP_SIZE     = 20        # DE population (fixed, not ×ndim)
MAX_GEN      = 50        # generations
N_ATTACK_IMG = 30        # images to attack
F  = 0.8                 # mutation factor
CR = 0.7                 # crossover rate

# Loss weights
W_OBJ = 1.0   # objectness suppression (primary goal)
W_FP  = 0.3   # false-positive penalty
W_TV  = 0.01  # total variation (smoothness)
W_NPS = 0.001 # non-printability score

print(f'Surrogate weights: {BEST_PT}  (exists={BEST_PT.exists()})')
print(f'Val images dir:    {VAL_IMAGES}')
print()

# ── Initialise attacker ───────────────────────────────────
attacker = DEAttackReal(
    model_path=str(BEST_PT),
    device='0',
    conf_threshold=0.25,
    iou_threshold=0.5,
    verbose=True,
)

# ── Select attack images ─────────────────────────────────
n_ready = attacker.prepare_attack_set(
    img_dir=str(VAL_IMAGES),
    lbl_dir=str(VAL_LABELS),
    n_images=N_ATTACK_IMG,
    seed=42,
)
assert n_ready >= 10, f'Only {n_ready} images with detections — need ≥10'

# ── Run DE optimisation ──────────────────────────────────
best_patch, history = attacker.run_de(
    patch_size=PATCH_SIZE,
    num_patches=NUM_PATCHES,
    pop_size=POP_SIZE,
    max_gen=MAX_GEN,
    F=F, CR=CR,
    w_obj=W_OBJ, w_fp=W_FP, w_tv=W_TV, w_nps=W_NPS,
)

# ── Save artefacts ───────────────────────────────────────
import numpy as np
ATTACK_DIR = WORK / 'attack_results'
ATTACK_DIR.mkdir(exist_ok=True)

np.save(str(ATTACK_DIR / 'best_patch.npy'), best_patch)
np.save(str(ATTACK_DIR / 'de_history.npy'), np.array(history))

plot_convergence(history, str(ATTACK_DIR / 'de_convergence.png'))
plot_patch(best_patch,   str(ATTACK_DIR / 'best_patch_vis.png'))

print(f'\nPatch shape: {best_patch.shape}, mean={best_patch.mean():.1f}, std={best_patch.std():.1f}')
print(f'Artefacts saved to {ATTACK_DIR}')


In [ ]:
# ── Cell 9: ★ Real Evaluation ★ ──────────────────────────
print('═' * 50)
print('  Evaluation on ATTACK set (in-sample)')
print('═' * 50)
metrics_attack = attacker.evaluate_patch(
    best_patch, num_patches=NUM_PATCHES,
)
for k, v in metrics_attack.items():
    print(f'  {k:>22s} : {v}')

# ── Out-of-sample evaluation ──
print()
print('═' * 50)
print('  Evaluation on HELD-OUT set (out-of-sample)')
print('═' * 50)
metrics_holdout = attacker.evaluate_patch(
    best_patch,
    img_dir=str(VAL_IMAGES),
    lbl_dir=str(VAL_LABELS),
    n_images=50,
    num_patches=NUM_PATCHES,
    seed=99,
)
for k, v in metrics_holdout.items():
    print(f'  {k:>22s} : {v}')

# ── Save side-by-side examples ────────────────────────────
save_attack_examples(
    attacker, best_patch,
    num_patches=NUM_PATCHES,
    save_dir=str(ATTACK_DIR / 'examples'),
    n=6,
)

print(f'\n✅ Phase 5 complete.')
print(f'   ASR (attack set)  = {metrics_attack["ASR"]:.2%}')
print(f'   ASR (held-out)    = {metrics_holdout["ASR"]:.2%}')
print(f'   AORR (held-out)   = {metrics_holdout["AORR"]:.4f}')


In [ ]:
# ── Cell 10: Display all results ─────────────────────────
from IPython.display import display, Image as IPImage
from pathlib import Path
import os

TRAIN_DIR  = WORK / 'runs/detect/surrogate_train'
ATTACK_DIR = WORK / 'attack_results'

# ── Training curves ──────────────────────────────────────
print('── Training Results ──')
for name in ['results.png', 'confusion_matrix.png',
             'BoxP_curve.png', 'BoxR_curve.png', 'BoxF1_curve.png']:
    p = TRAIN_DIR / name
    if p.exists():
        print(f'\n{name}:')
        display(IPImage(filename=str(p), width=700))

# ── Attack results ───────────────────────────────────────
print('\n── Attack Results ──')
for name in ['de_convergence.png', 'best_patch_vis.png']:
    p = ATTACK_DIR / name
    if p.exists():
        print(f'\n{name}:')
        display(IPImage(filename=str(p), width=600))

# ── Side-by-side attack examples ─────────────────────────
examples_dir = ATTACK_DIR / 'examples'
if examples_dir.exists():
    print('\n── Attack Examples (clean=green | patched=red) ──')
    for f in sorted(examples_dir.glob('*.jpg')):
        display(IPImage(filename=str(f), width=900))

print('\n✅ All results displayed.')
